# Phase 5: Saliency Metrics

This notebook runs the quantitative part of the stress test.

It compares saliency maps computed on:

- original images;
- images with approximate background perturbations.

Metrics:

- IoU between the top 20% most salient pixels;
- Spearman rank correlation between flattened saliency maps;
- confidence delta;
- prediction change flag.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import csv
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CANDIDATE_MANIFESTS = [
    PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv",
    PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest_debug.csv",
    PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest.csv",
]
MANIFEST = next((path for path in CANDIDATE_MANIFESTS if path.exists()), None)
if MANIFEST is None:
    raise FileNotFoundError("No AwA2 manifest found.")

CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"
CSV_OUTPUT = PROJECT_ROOT / "outputs" / "reports" / "phase5_saliency_metrics_notebook.csv"
FIGURE_OUTPUT = PROJECT_ROOT / "outputs" / "figures" / "phase5_saliency_comparison_notebook.png"

MAX_IMAGES = 4
XAI_METHODS = ["gradcam", "integrated_gradients"]
IG_STEPS = 16
TOP_PERCENT = 20
MASK_STRATEGY = "center_ellipse"
FOREGROUND_SCALE = 0.68
ALLOW_INCORRECT = False

print("manifest:", MANIFEST)
print("checkpoint:", CHECKPOINT)
print("checkpoint_exists:", CHECKPOINT.exists())
print("csv_output:", CSV_OUTPUT)
print("figure_output:", FIGURE_OUTPUT)

## Run Phase 5

For speed, use only `gradcam` first. Add `integrated_gradients` when the pipeline is working and you have enough GPU time.

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "run_phase5_metrics.py"),
    "--manifest", str(MANIFEST),
    "--checkpoint", str(CHECKPOINT),
    "--csv-output", str(CSV_OUTPUT),
    "--figure-output", str(FIGURE_OUTPUT),
    "--max-images", str(MAX_IMAGES),
    "--ig-steps", str(IG_STEPS),
    "--top-percent", str(TOP_PERCENT),
    "--mask-strategy", MASK_STRATEGY,
    "--foreground-scale", str(FOREGROUND_SCALE),
    "--xai-methods", *XAI_METHODS,
]
if ALLOW_INCORRECT:
    cmd.append("--allow-incorrect")

print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Phase 5 failed with exit code {completed.returncode}")

## Inspect CSV Metrics Intuitively

The CSV contains one row for each combination of image, XAI method, and perturbation. Instead of reading raw rows, we convert the file into three questions:

- **Did the prediction change?** This tells us whether the perturbation changed the model decision.
- **Did the confidence drop?** This tells us whether the model became less certain after the background changed.
- **Did the saliency map move?** Low IoU or low Spearman means the explanation is unstable even if the prediction is unchanged.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if not CSV_OUTPUT.exists():
    raise FileNotFoundError(CSV_OUTPUT)

metrics = pd.read_csv(CSV_OUTPUT)
iou_col = next(column for column in metrics.columns if column.startswith("iou_top_"))

numeric_cols = ["original_confidence", "perturbed_confidence", "confidence_delta", iou_col, "spearman"]
for column in numeric_cols:
    metrics[column] = pd.to_numeric(metrics[column], errors="coerce")

if metrics["prediction_changed"].dtype == object:
    metrics["prediction_changed"] = metrics["prediction_changed"].astype(str).str.lower().eq("true")

# The script column confidence_delta is perturbed - original.
# For interpretation, confidence_drop is more intuitive: positive means the model lost confidence.
metrics["confidence_drop"] = metrics["original_confidence"] - metrics["perturbed_confidence"]
metrics["saliency_drift"] = (1.0 - metrics[iou_col]) + (1.0 - metrics["spearman"]) / 2.0
metrics["prediction_transition"] = (
    metrics["original_prediction"].astype(str)
    + " -> "
    + metrics["perturbed_prediction"].astype(str)
)
metrics["case"] = (
    metrics["true_class"].astype(str)
    + " | "
    + metrics["prediction_transition"].astype(str)
    + " | "
    + metrics["perturbation"].astype(str)
    + " | "
    + metrics["xai_method"].astype(str)
)

print("rows:", len(metrics))
print("unique image indices:", metrics["index"].nunique())
print("xai methods:", sorted(metrics["xai_method"].unique()))
print("perturbations:", sorted(metrics["perturbation"].unique()))
print("prediction change rate:", round(metrics["prediction_changed"].mean(), 3))
metrics.head()


### Summary by XAI Method and Perturbation

Read this table as the experimental answer in compact form. The most suspicious cases have **low IoU**, **low Spearman**, and **positive confidence drop**.


In [ ]:
summary = (
    metrics.groupby(["xai_method", "perturbation"], as_index=False)
    .agg(
        examples=("index", "count"),
        prediction_change_rate=("prediction_changed", "mean"),
        mean_confidence_drop=("confidence_drop", "mean"),
        mean_iou=(iou_col, "mean"),
        std_iou=(iou_col, "std"),
        mean_spearman=("spearman", "mean"),
        std_spearman=("spearman", "std"),
        mean_saliency_drift=("saliency_drift", "mean"),
    )
    .sort_values(["mean_saliency_drift", "prediction_change_rate"], ascending=False)
)

summary.style.format({
    "prediction_change_rate": "{:.2%}",
    "mean_confidence_drop": "{:.3f}",
    "mean_iou": "{:.3f}",
    "std_iou": "{:.3f}",
    "mean_spearman": "{:.3f}",
    "std_spearman": "{:.3f}",
    "mean_saliency_drift": "{:.3f}",
}).background_gradient(subset=["mean_saliency_drift"], cmap="YlOrRd")


### Which Animal Appears After Perturbation?

This table makes the failure mode explicit: for each perturbation, it shows the animal predicted after the background was changed. The key column is `prediction_transition`, for example `antelope -> horse`.


In [ ]:
transition_summary = (
    metrics.groupby([
        "xai_method",
        "perturbation",
        "true_class",
        "original_prediction",
        "perturbed_prediction",
        "prediction_transition",
    ], as_index=False)
    .agg(
        rows=("index", "count"),
        prediction_changed=("prediction_changed", "max"),
        mean_confidence_drop=("confidence_drop", "mean"),
        mean_saliency_drift=("saliency_drift", "mean"),
    )
    .sort_values(["prediction_changed", "rows", "mean_saliency_drift"], ascending=[False, False, False])
)

transition_summary.style.format({
    "mean_confidence_drop": "{:.3f}",
    "mean_saliency_drift": "{:.3f}",
}).background_gradient(subset=["mean_saliency_drift"], cmap="YlOrRd")


### Visual Inspection of the Metrics

- **High IoU** means the top 20% salient region stayed in roughly the same place.
- **High Spearman** means the full pixel ranking stayed similar.
- **Positive confidence drop** means the model became less confident after background perturbation.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

for ax, value, title, ylabel in [
    (axes[0], "mean_iou", "Top-saliency overlap", "Mean IoU"),
    (axes[1], "mean_spearman", "Full-map rank stability", "Mean Spearman"),
    (axes[2], "mean_confidence_drop", "Prediction confidence loss", "Mean confidence drop"),
]:
    pivot = summary.pivot(index="perturbation", columns="xai_method", values=value)
    pivot.plot(kind="bar", ax=ax, rot=20)
    ax.set_title(title)
    ax.set_xlabel("Perturbation")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="XAI method")

plt.tight_layout()
plt.show()


### IoU vs Spearman: Stable or Unstable Explanations?

The upper-right area is good: the explanation stayed similar. The lower-left area is suspicious: the saliency map changed both in its top support and in its global ranking.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.8))
for method, subset in metrics.groupby("xai_method"):
    ax.scatter(subset[iou_col], subset["spearman"], s=70, alpha=0.75, label=method)

ax.axvline(0.5, color="tab:orange", linestyle="--", linewidth=1)
ax.axhline(0.5, color="tab:orange", linestyle="--", linewidth=1)
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-1.03, 1.03)
ax.set_xlabel("IoU of top-saliency region")
ax.set_ylabel("Spearman rank correlation")
ax.set_title("Saliency stability after background perturbation")
ax.grid(alpha=0.25)
ax.legend(title="XAI method")
plt.show()


### Most Unstable Examples

These rows are the best candidates for the blog post: they show where the explanation moved the most. A high `saliency_drift` means low IoU and/or low Spearman.


In [ ]:
most_unstable = metrics.sort_values("saliency_drift", ascending=False).head(12)
display_cols = [
    "true_class",
    "original_prediction",
    "perturbed_prediction",
    "prediction_transition",
    "xai_method",
    "perturbation",
    iou_col,
    "spearman",
    "confidence_drop",
    "saliency_drift",
    "prediction_changed",
]
most_unstable[display_cols].style.format({
    iou_col: "{:.3f}",
    "spearman": "{:.3f}",
    "confidence_drop": "{:.3f}",
    "saliency_drift": "{:.3f}",
}).background_gradient(subset=["saliency_drift"], cmap="YlOrRd")


### Save the Intuitive Metric Report

This cell saves the aggregated table and the most unstable examples so they can be reused in the HTML blog post.


In [ ]:
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = REPORT_DIR / "phase5_metric_summary_notebook.csv"
unstable_path = REPORT_DIR / "phase5_most_unstable_examples_notebook.csv"
transitions_path = REPORT_DIR / "phase5_prediction_transitions_notebook.csv"
summary.to_csv(summary_path, index=False)
most_unstable.to_csv(unstable_path, index=False)
transition_summary.to_csv(transitions_path, index=False)
print("saved:", summary_path)
print("saved:", unstable_path)
print("saved:", transitions_path)


## Inspect Visual Comparison

The figure uses one saliency method, preferring Grad-CAM when available, and compares original vs perturbed heatmaps.

Each perturbed panel should now show an overlay like `pred after: original_animal -> perturbed_animal`. If you do not see it, rerun the **Run Phase 5** cell above so the PNG is regenerated with the updated plotting code.


In [ ]:
if FIGURE_OUTPUT.exists():
    print("visual comparison:", FIGURE_OUTPUT)
    print("If the panels do not show `pred after: original -> perturbed`, rerun the Run Phase 5 cell to regenerate this PNG.")
    display(Image(filename=str(FIGURE_OUTPUT)))
else:
    print("Figure was not created:", FIGURE_OUTPUT)
